In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import re

In [8]:
# Config
REAL_BRANDS = ['XIAOMI','SAMSUNG','POCO','INFINIX','TECNO','ITEL','REALME',
               'OPPO','VIVO','NOKIA','APPLE','REDMI','HUAWEI','OUKITEL',
               'BLACKVIEW','REDBEAT','OALE','RUIOO']

In [9]:

def get_driver():
    """Initialize Chrome driver"""
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    return webdriver.Chrome(options=options)

In [10]:
def extract_product(article):
    """Extract data from a single product article"""
    try:
        # Name
        try:
            name = article.find_element(By.CSS_SELECTOR, "h3.name").text.strip()
        except:
            name = article.text.split("\n")[0].strip()

        # Price
        price_text = article.find_element(By.CSS_SELECTOR, "div.prc").text
        price_num = int(re.sub(r'[^0-9]', '', price_text))

        # Old price (optional)
        try:
            old_price_text = article.find_element(By.CSS_SELECTOR, "div.old").text
            old_price_num = int(re.sub(r'[^0-9]', '', old_price_text))
        except:
            old_price_text, old_price_num = None, None

        # Discount (optional)
        try:
            discount = article.find_element(By.CSS_SELECTOR, "div.bdg._dsct").text
        except:
            discount = None

        # Brand & URL
        brand = article.get_attribute("data-brand") or name.split()[0]
        try:
            url = article.find_element(By.CSS_SELECTOR, "a.core").get_attribute("href")
        except:
            url = None

        return {
            "name": name, "brand": brand.upper(),
            "price": price_text, "price_num": price_num,
            "old_price": old_price_text, "old_price_num": old_price_num,
            "discount": discount, "url": url
        }
    except:
        return None


In [11]:
def scrape_jumia(search_term="smartphones", pages=5):
    """Scrape Jumia for N pages"""
    driver = get_driver()
    all_data = []
    try:
        for page in range(1, pages + 1):
            url = f"https://www.jumia.com.ng/catalog/?q={search_term}&page={page}#catalog-listing"
            print(f"Scraping page {page}")
            driver.get(url)
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "article.prd"))
            )
            time.sleep(2)
            articles = driver.find_elements(By.CSS_SELECTOR, "article.prd")
            for art in articles:
                data = extract_product(art)
                if data:
                    all_data.append(data)
    finally:
        driver.quit()
    return pd.DataFrame(all_data)

In [12]:
def clean_data(df):
    """Clean and filter to real smartphones only"""
    df["brand"] = df["brand"].str.upper().str.strip()
    df_clean = df[df["brand"].isin(REAL_BRANDS)].copy()
    df_clean = df_clean[(df_clean["price_num"] >= 40000) & (df_clean["price_num"] <= 1500000)]
    df_clean = df_clean[df_clean["name"].str.len() > 20]
    df_clean = df_clean.drop_duplicates(subset=["name", "price_num"])
    return df_clean



In [13]:
# --- Main ---
if __name__ == "__main__":
    df_raw = scrape_jumia("smartphones", pages=5)
    df_final = clean_data(df_raw)
    print(f"Clean: {len(df_final)} smartphones")
    print(df_final["brand"].value_counts())
    df_final.to_excel("jumia_smartphones_portfolio.xlsx", index=False)

Scraping page 1
Scraping page 2
Scraping page 3
Scraping page 4
Scraping page 5
Clean: 123 smartphones
brand
SAMSUNG      46
XIAOMI       17
OUKITEL      14
APPLE        10
ITEL          7
INFINIX       6
HUAWEI        5
OPPO          4
RUIOO         3
REDBEAT       2
BLACKVIEW     2
POCO          2
TECNO         2
REDMI         1
VIVO          1
OALE          1
Name: count, dtype: int64
